# Validation — `kappa-lora-spectral-targeting`

**What this measures:** The claim ("spectral targeting halves trainable params without losing fit vs standard LoRA") is exactly what this repo's own MetaMathQA harness measures, so the target metric is the harness-emitted `num_trainable_params` for a new `lora/llama-3.2-3B-rank32-kappa05` config (`condition_number_top_fraction: 0.5`, the default-off flag turned ON), compared against the published LoRA r=32 corpus row: top-50% spectral selection keeps ceil(56/2)=28 of 56 matched q/v modules, so trainable params must drop to ≈0.5× the 9,175,040 baseline (28 layers × [32·(3072+3072) + 32·(3072+1024)]), while the guardrail `test_accuracy` stays within the repo's own ±0.02 parity band (verification notebook) so fit is not lost. The baseline is the published corpus row — the new config key is a TypeError on `main`, so no baseline arm can run it.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`cedb941c3dee`](https://github.com/mayorquinmachines/peft/commit/cedb941c3deef8d6deacb9f46a0c21dfc984afc7)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa05` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "cedb941c3deef8d6deacb9f46a0c21dfc984afc7"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa05` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa05/adapter_config.json`:

```json
{
  "peft_type": "LORA",
  "task_type": "CAUSAL_LM",
  "base_model_name_or_path": "meta-llama/Llama-3.2-3B",
  "revision": "main",
  "inference_mode": false,
  "target_modules": ["q_proj", "v_proj"],
  "r": 32,
  "lora_alpha": 64,
  "condition_number_top_fraction": 0.5
}
```

In [ ]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa05/adapter_config.json")).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa05` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa05/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa05"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy", "total_time"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5046272,
        "baseline": 9175040
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.44,
        "baseline": 0.46
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": 2700,
        "baseline": null
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5046272 with `test_accuracy` >= 0.44, `total_time` <= 2700 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
benchmarks:
  - name: kappa-lora-spectral-targeting
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/lora/llama-3.2-3B-rank32-kappa05
        # PR branch (non-main) writes to temporary_results/ per the verification notebook
        results_glob: "method_comparison/MetaMathQA/temporary_results/lora--llama-3.2-3B-rank32-kappa05--*.json"
        method: lora
        smoke:
          params_path: method_comparison/MetaMathQA/default_training_params.json
          overrides:
            num_steps: 50
    scorer: num_trainable_params
    metrics:
      - name: num_trainable_params
        direction: min
        # 0.55 x 9,175,040 baseline: top_fraction=0.5 keeps ceil(56/2)=28 of 56 matched modules
        # (q_proj 196,608 params, v_proj 131,072); balanced pick = 4,587,520 (exactly 0.50x),
        # worst all-q pick = 5,505,024 (0.60x); 5,046,272 certifies ~halving with split skew margin.
        threshold: 5046272
        role: target
      - name: test_accuracy
        direction: max
        # floor = published lora r=32 row (0.46) - 0.02, the parity band the repo's own
        # verification notebook uses for accuracy; baseline 0.46 clears it by construction.
        threshold: 0.44
        role: guardrail
      - name: total_time
        direction: min
        # notebook: supra r=32 = 26.1 min (1566 s) on A100; 2700 s adds model download + GSM8K eval headroom. Reported, never a fit gate.
        threshold: 2700
        role: cost
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        # 28 layers x (32x(3072+3072) + 32x(3072+1024)) = 9,175,040; cross-checked: supra r32 row
        # trainable 12,698,224 = 9,175,040 (LoRA A/B) + 28 x 125,828 (sparse), confirming q/v targeting.
        num_trainable_params: 9175040
        test_accuracy: 0.46
    compute:
      # notebook: 5000 steps + GSM8K eval ~= 26 min on A100 with model cached; 4500 s covers
      # cold-start download of gated Llama-3.2-3B plus evaluation, per arm.
      tier: gpu
      timeout_s: 4500
    held_constant:
      - "same base model meta-llama/Llama-3.2-3B and tokenizer as the published lora rank32 row"
      - "same rank r=32 and target_modules q_proj,v_proj so only spectral targeting differs"
      - "same training protocol via the harness default_training_params.json (5000 steps, GSM8K valid/test eval)"
      - "same harness run.py invocation as the verification notebook (--verbose --clean <experiment>)"
    avoid:
      - "unpinned base-model revision on the gated meta-llama repo"
      - "adding a training_params.json override (breaks comparability with the published corpus rows)"
      - "comparing total_time across GPU types or using it to judge fit"
    provenance:
      num_trainable_params: "user_guidance: spectral targeting halves trainable params vs standard LoRA; maintainer_comment:@mayorquinmachines asks contributors to show far fewer trainable params than standard LoRA on MetaMathQA"
      test_accuracy: "user_guidance: without losing fit; parity band of 0.02 from user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
      total_time: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing (26.1 min reference for a 5000-step r=32 run)"
      baseline: "published_corpus:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json with num_trainable_params cross-derived from Llama-3.2-3B geometry (9,175,040) and confirmed by the supra r32 reference row 12,698,224 = 9,175,040 + 28 x 125,828; re-read exact values from the corpus file before scoring"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py invoked exactly as in user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
      experiments: "inferred from the PR diff: LoraConfig field condition_number_top_fraction=0.5 applied to the repo's lora rank32 protocol (r=32, q_proj+v_proj)"
      smoke: "inferred: default_training_params.json is the harness default protocol file; num_steps mirrors the notebook's 5000-step run truncated to 50 steps"

loop:
  max_iterations: 8
  fix_code: true
```